In [3]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import Sequential, layers, regularizers                       # regularizers -- adding regularization into the model (L1, and L2)
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
train_dir = "/kaggle/input/datasets/ayush1220/cifar10/cifar10/train"
test_dir  = "/kaggle/input/datasets/ayush1220/cifar10/cifar10/test"

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(32,32),
    batch_size=64,
    shuffle=True
)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(32,32),
    batch_size=64,
    shuffle=True
)

Found 50000 files belonging to 10 classes.


I0000 00:00:1789914136.569122      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789914136.571266      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 10000 files belonging to 10 classes.


In [5]:
x_train_list = []
y_train_list = []

x_test_list = []
y_test_list = []


for imgs, labels in train_ds:
    x_train_list.append(imgs.numpy())             # being converted into numpy arrays, and then being append into the list
    y_train_list.append(labels.numpy())

for imgs, labels in test_ds:
    x_test_list.append(imgs.numpy())
    y_test_list.append(labels.numpy())

In [9]:
print(len(x_train_list))

print(len(x_test_list))

print(x_train_list[0])

782
157
[[[[207. 242. 255.]
   [206. 242. 255.]
   [189. 220. 233.]
   ...
   [212. 240. 255.]
   [208. 238. 255.]
   [211. 240. 255.]]

  [[204. 238. 255.]
   [203. 238. 255.]
   [189. 220. 231.]
   ...
   [184. 212. 227.]
   [208. 236. 254.]
   [207. 235. 255.]]

  [[205. 239. 255.]
   [203. 238. 255.]
   [191. 223. 235.]
   ...
   [190. 218. 232.]
   [207. 236. 253.]
   [208. 237. 255.]]

  ...

  [[ 89.  89.  78.]
   [ 99. 100.  83.]
   [114. 115.  97.]
   ...
   [ 86.  91.  80.]
   [ 82.  86.  74.]
   [112. 114. 101.]]

  [[ 90.  92.  79.]
   [ 98. 100.  87.]
   [109. 111.  98.]
   ...
   [ 77.  78.  71.]
   [ 75.  76.  70.]
   [ 80.  81.  74.]]

  [[ 97. 106.  99.]
   [105. 114. 106.]
   [105. 114. 107.]
   ...
   [ 94.  95.  90.]
   [112. 113. 108.]
   [106. 107. 101.]]]


 [[[119. 137.  84.]
   [ 96. 113.  70.]
   [ 42.  57.  26.]
   ...
   [ 56.  44.  53.]
   [ 59.  46.  56.]
   [ 70.  56.  65.]]

  [[127. 142.  96.]
   [120. 134.  96.]
   [ 70.  83.  56.]
   ...
   [ 63.  50.

In [19]:
# preparing final datasets

x_train = np.concatenate(x_train_list, axis=0)           # concatenating all the lists in the list, and it returns an array after concatenating
y_train = np.concatenate(y_train_list, axis=0)

x_test = np.concatenate(x_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

In [20]:
print(type(x_train_list))
print(type(x_train))
print()
print(x_train[0])

<class 'list'>
<class 'numpy.ndarray'>

[[[207. 242. 255.]
  [206. 242. 255.]
  [189. 220. 233.]
  ...
  [212. 240. 255.]
  [208. 238. 255.]
  [211. 240. 255.]]

 [[204. 238. 255.]
  [203. 238. 255.]
  [189. 220. 231.]
  ...
  [184. 212. 227.]
  [208. 236. 254.]
  [207. 235. 255.]]

 [[205. 239. 255.]
  [203. 238. 255.]
  [191. 223. 235.]
  ...
  [190. 218. 232.]
  [207. 236. 253.]
  [208. 237. 255.]]

 ...

 [[ 89.  89.  78.]
  [ 99. 100.  83.]
  [114. 115.  97.]
  ...
  [ 86.  91.  80.]
  [ 82.  86.  74.]
  [112. 114. 101.]]

 [[ 90.  92.  79.]
  [ 98. 100.  87.]
  [109. 111.  98.]
  ...
  [ 77.  78.  71.]
  [ 75.  76.  70.]
  [ 80.  81.  74.]]

 [[ 97. 106.  99.]
  [105. 114. 106.]
  [105. 114. 107.]
  ...
  [ 94.  95.  90.]
  [112. 113. 108.]
  [106. 107. 101.]]]


In [21]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [22]:
print(x_train.shape)
print(x_test.shape)

(50000, 32, 32, 3)
(10000, 32, 32, 3)


| Axis | Size | What it Represents |
| :--- | :--- | :--- |
| **Axis 0** | **`50,000`** | **Number of samples (images):** There are 50,000 individual pictures. |
| **Axis 1** | **`32`** | **Height:** 32 pixels vertically (rows). |
| **Axis 2** | **`32`** | **Width:** 32 pixels horizontally (columns). |
| **Axis 3** | **`3`** | **Color Channels:** 3 color planes — **Red, Green, and Blue (RGB)**. |

### L1 Regularization

In [23]:
lr_l1 = regularizers.l1(1e-4)

# STEP 1
model = keras.Sequential([
    keras.Input(shape=(32,32,3)),
    layers.Conv2D(32, kernel_size=(3,3), padding='valid', activation='relu', kernel_regularizer=lr_l1),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.Conv2D(64, kernel_size=(3,3), activation='relu', kernel_regularizer=lr_l1),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.Conv2D(128, kernel_size=(3,3), activation='relu', kernel_regularizer=lr_l1),
    layers.Flatten(),
    layers.Dense(62, activation='relu', kernel_regularizer=lr_l1),
    layers.Dense(10)
])


# STEP 2
model.compile(
    loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer = keras.optimizers.Adam(3e-4),
    metrics = ['accuracy']
)


# STEP 3
early_stop = EarlyStopping(
    monitor = 'val_loss',            # metrics to monitor
    patience = 3,                    # stop if no improvement for 3 consecutive epochs
    restore_best_weights = True      # automatically restore weights from the best epochs
)


# STEP 4
model.fit(
    x_train, y_train,
    batch_size = 64, epochs=10,
    validation_split = 0.3,
    callbacks = [early_stop],
    verbose = 2
)


# STEP 5
test_loss, test_acc = model.evaluate(
    x_test, y_test,
    batch_size = 64,
    verbose = 2
)


Epoch 1/10


I0000 00:00:1789918888.647294     328 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


547/547 - 13s - 23ms/step - accuracy: 0.3331 - loss: 2.1146 - val_accuracy: 0.4428 - val_loss: 1.7791
Epoch 2/10
547/547 - 3s - 5ms/step - accuracy: 0.4597 - loss: 1.7298 - val_accuracy: 0.4839 - val_loss: 1.6724
Epoch 3/10
547/547 - 3s - 5ms/step - accuracy: 0.4910 - loss: 1.6324 - val_accuracy: 0.5177 - val_loss: 1.5678
Epoch 4/10
547/547 - 3s - 5ms/step - accuracy: 0.5186 - loss: 1.5690 - val_accuracy: 0.5365 - val_loss: 1.5285
Epoch 5/10
547/547 - 3s - 5ms/step - accuracy: 0.5417 - loss: 1.5101 - val_accuracy: 0.5421 - val_loss: 1.4894
Epoch 6/10
547/547 - 3s - 5ms/step - accuracy: 0.5604 - loss: 1.4612 - val_accuracy: 0.5662 - val_loss: 1.4469
Epoch 7/10
547/547 - 3s - 5ms/step - accuracy: 0.5764 - loss: 1.4228 - val_accuracy: 0.5908 - val_loss: 1.3891
Epoch 8/10
547/547 - 3s - 5ms/step - accuracy: 0.5898 - loss: 1.3851 - val_accuracy: 0.5838 - val_loss: 1.4148
Epoch 9/10
547/547 - 3s - 5ms/step - accuracy: 0.6022 - loss: 1.3557 - val_accuracy: 0.5927 - val_loss: 1.3769
Epoch 10/1

### L2 Regularization

In [24]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

# STEP 1
l2_reg = regularizers.l2(1e-4)  

model_l2 = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding='valid', activation='relu', kernel_regularizer=l2_reg),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, 3, activation='relu', kernel_regularizer=l2_reg),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', kernel_regularizer=l2_reg),
    layers.Flatten(),
    layers.Dense(64, activation='relu', kernel_regularizer=l2_reg),
    layers.Dense(10)
])


# STEP 2
model_l2.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(3e-4),
    metrics=['accuracy']
)


# STEP 3
early_stop = EarlyStopping(
    monitor='val_loss',        
    patience=3,                
    restore_best_weights=True   
)


# STEP 4
model_l2.fit(
    x_train, y_train,
    batch_size=64,
    epochs=25,                  
    validation_split=0.2,     
    callbacks=[early_stop],
    verbose=2
)

# STEP 5
test_loss, test_acc = model_l2.evaluate(
    x_test, y_test,
    batch_size=64,
    verbose=2
)
print(f"L2 Model Test Accuracy: {test_acc * 100:.2f}%")

Epoch 1/25
625/625 - 8s - 12ms/step - accuracy: 0.3647 - loss: 1.7466 - val_accuracy: 0.4685 - val_loss: 1.5020
Epoch 2/25
625/625 - 3s - 4ms/step - accuracy: 0.4951 - loss: 1.4156 - val_accuracy: 0.5380 - val_loss: 1.3276
Epoch 3/25
625/625 - 3s - 4ms/step - accuracy: 0.5426 - loss: 1.3023 - val_accuracy: 0.5669 - val_loss: 1.2516
Epoch 4/25
625/625 - 3s - 4ms/step - accuracy: 0.5781 - loss: 1.2158 - val_accuracy: 0.5919 - val_loss: 1.1843
Epoch 5/25
625/625 - 3s - 4ms/step - accuracy: 0.6076 - loss: 1.1413 - val_accuracy: 0.6118 - val_loss: 1.1336
Epoch 6/25
625/625 - 3s - 4ms/step - accuracy: 0.6276 - loss: 1.0929 - val_accuracy: 0.6313 - val_loss: 1.0845
Epoch 7/25
625/625 - 3s - 4ms/step - accuracy: 0.6481 - loss: 1.0346 - val_accuracy: 0.6480 - val_loss: 1.0400
Epoch 8/25
625/625 - 3s - 4ms/step - accuracy: 0.6671 - loss: 0.9850 - val_accuracy: 0.6524 - val_loss: 1.0276
Epoch 9/25
625/625 - 3s - 4ms/step - accuracy: 0.6802 - loss: 0.9477 - val_accuracy: 0.6661 - val_loss: 0.9973
